# HDGT Phase 1.6 — MP-DocVQA Dataset Exploration

This notebook walks through:
1. Dataset format — QA annotations, page IDs, answer structure
2. Statistics — splits, pages per context, question length
3. HDGT graph loading — node types, edge types, metadata
4. BM25 retrieval — end-to-end example on one question
5. Mock graph traversal — l-hop BFS on the HDGT graph

**Requires**: `qas.zip` in `data/MP-DocVQA/`. Sections 3–5 additionally
require compiled graphs in `experiments/mpdocvqa/`.

In [ ]:
import sys, json, zipfile, re
from pathlib import Path
from collections import Counter

# Add project root to path
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Workstation path — change if running locally
DATA_ROOT   = Path("data/MP-DocVQA")
GRAPHS_DIR  = Path("experiments/mpdocvqa")

print(f"Project root : {PROJECT_ROOT}")
print(f"Data root    : {DATA_ROOT.resolve()}")
print(f"Graphs dir   : {GRAPHS_DIR.resolve()}")

---
## Section 1: Dataset Format

In [ ]:
# Load val split directly from the zip
zip_path = DATA_ROOT / "qas.zip"
with zipfile.ZipFile(zip_path, "r") as zf:
    with zf.open("val.json") as f:
        val_data = json.load(f)["data"]

print(f"Val split: {len(val_data):,} questions")
print("\nAvailable fields:", list(val_data[0].keys()))
print("\n--- Example (single-page question) ---")
print(json.dumps(val_data[0], indent=2))

In [ ]:
# Show a multi-page question
multi_page = [x for x in val_data if len(x["page_ids"]) > 5]
print(f"Questions with >5 pages in context: {len(multi_page)}")
print("\n--- Example (multi-page question) ---")
print(json.dumps(multi_page[0], indent=2))

---
## Section 2: Dataset Statistics

In [ ]:
from hdgt.evaluation.loaders import MPDocVQALoader

loader = MPDocVQALoader(DATA_ROOT, split="val")

page_counts = []
q_lengths   = []
ans_lengths = []

for item in loader:
    page_counts.append(len(item["page_ids"]))
    q_lengths.append(len(item["question"].split()))
    if item["answers"]:
        ans_lengths.append(len(item["answers"][0]))

print(f"Questions  : {len(page_counts):,}")
print(f"Pages/ctx  : avg={sum(page_counts)/len(page_counts):.2f}  max={max(page_counts)}")
print(f"Q length   : avg={sum(q_lengths)/len(q_lengths):.1f} words")
print(f"Ans length : avg={sum(ans_lengths)/len(ans_lengths):.1f} chars")

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].hist(page_counts, bins=20, color='steelblue', edgecolor='white')
    axes[0].set_title('Pages per Context')
    axes[0].set_xlabel('Number of pages')
    axes[0].set_ylabel('Count')
    
    axes[1].hist(q_lengths, bins=20, color='darkorange', edgecolor='white')
    axes[1].set_title('Question Length (words)')
    axes[1].set_xlabel('Words')
    
    axes[2].hist(ans_lengths, bins=20, color='seagreen', edgecolor='white')
    axes[2].set_title('Answer Length (chars)')
    axes[2].set_xlabel('Characters')
    
    plt.tight_layout()
    plt.savefig('phase1_6_results/mpdocvqa_distributions.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Figure saved to phase1_6_results/mpdocvqa_distributions.png")
except ImportError:
    print("matplotlib not available — skipping plot.")

---
## Section 3: HDGT Graph Loading
> Requires compiled graphs in `experiments/mpdocvqa/`. Run `build_mpdocvqa_graphs.py` first.

In [ ]:
from hdgt.evaluation.loaders import GraphLoader, graph_to_node_list

graph_loader = GraphLoader(GRAPHS_DIR)
available = graph_loader.available_ids()
print(f"Compiled graphs available: {graph_loader.count():,}")
if available:
    print(f"Example IDs: {available[:3]}")

In [ ]:
# Load a graph and inspect it
if available:
    sample_id = available[0]
    graph = graph_loader.load(sample_id)
    
    print(f"Context ID : {sample_id}")
    print(f"Node types : {graph.node_types}")
    print(f"Edge types : {[et[1] for et in graph.edge_types]}")
    
    for ntype in graph.node_types:
        n = graph[ntype].x.shape[0]
        print(f"  {ntype:<10} : {n:>4} nodes")
    
    # Show node list
    nodes = graph_to_node_list(graph)
    print(f"\nTotal nodes (flat): {len(nodes)}")
    print("\nFirst 3 nodes:")
    for n in nodes[:3]:
        print(f"  [{n['type']:8}] page={n['page']}  content={n['content'][:60]!r}")
else:
    print("No compiled graphs found. Run build_mpdocvqa_graphs.py on the workstation first.")

---
## Section 4: BM25 Retrieval Example

In [ ]:
from hdgt.evaluation.retriever import BM25Retriever
from hdgt.evaluation.metrics import compute_metrics

if available:
    # Find a question whose context matches our loaded graph
    target_id = sample_id
    target_qa = None
    for item in MPDocVQALoader(DATA_ROOT, split="val"):
        ctx_id = loader.build_context_id(item["page_ids"])
        if ctx_id == target_id:
            target_qa = item
            break
    
    if target_qa:
        print(f"Question : {target_qa['question']}")
        print(f"Answers  : {target_qa['answers']}")
        print(f"Answer page: {target_qa['answer_page_idx']}")
        
        retriever = BM25Retriever()
        results = retriever.retrieve(target_qa["question"], graph, top_k=5)
        
        print("\n--- Top-5 BM25 Results ---")
        for i, r in enumerate(results, 1):
            print(f"  #{i} [{r['type']:8}] page={r['page']}  score={r['score']:.3f}")
            print(f"       {r['content'][:80]!r}")
        
        metrics = compute_metrics(
            retrieved_pages=[r["page"] for r in results],
            retrieved_contents=[r["content"] for r in results],
            ground_truth_page=target_qa["answer_page_idx"],
            ground_truth_answers=target_qa["answers"],
        )
        print("\n--- Metrics ---")
        for k, v in metrics.items():
            print(f"  {k:<12}: {v:.4f}")
    else:
        print("No matching question found for this context.")
else:
    print("No graphs available. Skipping retrieval demo.")

---
## Section 5: Mock Graph Traversal Example

In [ ]:
from hdgt.evaluation.retriever import MockGraphRetriever

if available and target_qa:
    mock_retriever = MockGraphRetriever(k_anchor=3, l_hops=2)
    mock_results   = mock_retriever.retrieve(target_qa["question"], graph, top_k=5)
    
    print("--- Top-5 Mock Graph Traversal Results ---")
    for i, r in enumerate(mock_results, 1):
        print(f"  #{i} [{r['type']:8}] page={r['page']}  score={r['score']:.4f}")
        print(f"       {r['content'][:80]!r}")
    
    mock_metrics = compute_metrics(
        retrieved_pages=[r["page"] for r in mock_results],
        retrieved_contents=[r["content"] for r in mock_results],
        ground_truth_page=target_qa["answer_page_idx"],
        ground_truth_answers=target_qa["answers"],
    )
    print("\n--- Metrics (Mock Graph) ---")
    for k, v in mock_metrics.items():
        print(f"  {k:<12}: {v:.4f}")
else:
    print("No graphs available. Skipping mock traversal demo.")